In [7]:
import numpy as np
import matplotlib.pyplot as plt
import os

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

from week3_trading_env import ToyTradingEnv


class EpisodeRewardLogger(BaseCallback):

    def __init__(self):

        super().__init__()

        self.episode_rewards = []

        self.current_reward = 0.0


    def _on_step(self):

        self.current_reward += self.locals["rewards"][0]

        if self.locals["dones"][0]:

            self.episode_rewards.append(
                self.current_reward
            )

            self.current_reward = 0.0

        return True



def train_ppo_on_trading(
    total_timesteps=100000
):

    env = ToyTradingEnv()

    callback = EpisodeRewardLogger()


    model = PPO(
        "MlpPolicy",
        env,
        verbose=0,
        learning_rate=3e-4,
        n_steps=512,
        batch_size=64,
        gamma=0.99
    )


    model.learn(
        total_timesteps=total_timesteps,
        callback=callback
    )


    env.close()


    return model, callback.episode_rewards



def moving_average(
    x,
    window=20
):

    if len(x) < window:

        return np.array(x)


    return np.convolve(
        x,
        np.ones(window)/window,
        mode="valid"
    )



def plot_rewards(
    rewards,
    filename="plots/ppo_trading_rewards.png"
):

    os.makedirs(
        "plots",
        exist_ok=True
    )


    rewards = np.array(rewards)


    plt.figure(
        figsize=(9,4)
    )


    plt.plot(
        rewards,
        alpha=0.25,
        label="Episode reward"
    )


    plt.plot(
        moving_average(rewards),
        label="20 episode moving average"
    )


    plt.xlabel("Episode")

    plt.ylabel("Reward")

    plt.title(
        "PPO on ToyTradingEnv"
    )


    plt.legend()

    plt.tight_layout()


    plt.savefig(filename)

    plt.close()


    print(
        f"Saved to {filename}"
    )



if __name__ == "__main__":


    model, rewards = train_ppo_on_trading()


    plot_rewards(
        rewards
    )


    np.save(
        "ppo_trading_returns.npy",
        np.array(rewards)
    )


    print(
        f"Final 20 episode average: {np.mean(rewards[-20:]):.2f}"
    )

C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\gymnasium\utils\env_checker.py:373: UserWarning: WARN: `check_env(warn=...)` parameter is now ignored.
  logger.warn("`check_env(warn=...)` parameter is now ignored.")
C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\gymnasium\utils\env_checker.py:317: UserWarning: WARN: A Box observation space minimum value is -infinity. This is probably too low.
  logger.warn(
C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\gymnasium\utils\env_checker.py:321: UserWarning: WARN: A Box observation space maximum value is infinity. This is probably too high.
  logger.warn(
C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\gymnasium\utils\env_checker.py:440: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(


Environment check passed
Episode  1: total reward =   -3.23
Episode  2: total reward =    8.86
Episode  3: total reward =   -4.49
Episode  4: total reward =  -12.83
Episode  5: total reward =   -6.85
Episode  6: total reward =  -10.57
Episode  7: total reward =   -3.69
Episode  8: total reward =   13.07
Episode  9: total reward =  -11.08
Episode 10: total reward =    6.03

Mean over 10 episodes: -2.48
Saved to plots/ppo_trading_rewards.png
Final 20 episode average: -3.13
